# **Laboratorio 9 - Visión por Computadora**

- Paula Barillas - 22764
- Gerardo Pineda - 22880
- Mónica Salvatierra - 22249

Link del repositorio: https://github.com/paulabaal12/LAB9-VCP

## **Task 2**

**En el pabellón principal del evento, se está llevando a cabo un concurso de cosplay. De repente, un grupo de 15 personas haciendo cosplay de Naruto entran al escenario exactamente iguales y posan muy juntos (simulando el Kage Bunshin no Jutsu o Jutsu Clones de Sombra). Las cámaras registran a los clones abrazados, superpuestos y parcialmente ocluidos unos detrás de otros. Usted nota que su modelo clásico basado en YOLOv8 falla rotundamente: la red neuronal sí encuentra a los 15 clones y dibuja cientos de cajas, pero al final en la pantalla solo aparecen 4 o 5 personas detectadas. El resto "desaparece". Considerando esto, responda:**

**1. Explique matemáticamente, utilizando la fórmula del IoU (Intersección sobre Unión), por qué el algoritmo NMS está causando que los clones superpuestos desaparezcan (Falsos Negativos).**

- En este escenario, los 15 clones están parados muy juntos y sus bounding boxes se superponen de manera masiva. El IoU entre dos cajas se calcula como: 

    $$IoU(A, B) = \frac{|A \cap B|}{|A \cup B|}$$


    Cuando dos personas están casi encima una de la otra, el área de intersección es muy grande en relación al área de unión, lo que da un IoU relativamente alto. NMS interpreta eso como si las dos cajas están detectando el mismo objeto y elimina la de menor confianza. El problema es que no eran el mismo objeto, realmente eran personas distintas. Los falsos negativos se generan al descartar detecciones reales pensando que son duplicados. 

**2. Si usted es el ingeniero a cargo y solo puede modificar los hiperparámetros en el código durante la inferencia: ¿Qué pasaría si ajusta el umbral de IoU del NMS a 0.15? ¿Qué pasaría si lo ajusta a 0.95? Justifique qué valor sería más adecuado para este problema de alta densidad.**

- Con un umbral de 0.15, NMS se vuelve extremadamente agresivo. Cualquier par de cajas que se solape aunque sea un poco se considera duplicado y se elimina. En un escenario como este, casi todas las cajas se descartan entre sí, y el resultado sería detectar incluso menos personas que antes.

    Con un umbral de 0.95, NMS solo elimina cajas que sean prácticamente idénticas (superpuestas en un 95%). Esto permitiría que coexistan detecciones de personas distintas aunque estén muy juntas, porque sus cajas, aunque se solapen bastante, raramente llegan a ese nivel de coincidencia.

    Para este problema en específico, el valor más adecuado sería uno alto, alrededor de 0.8-0.85. La idea es mantener las detecciones de personas cercanas y descartar únicamente las cajas que sean casi idénticas entre sí.

**3. Si el presupuesto le permitiera cambiar el modelo a YOLOv10, explique cómo su arquitectura Asignación Dual de Etiquetas (Dual Label Assignment) resolvería este problema de oclusión sin necesidad de tocar el NMS**

- YOLOv10 resuelve el problema con su arquitectura de Dual Label Assignment, utilizando dos cabezas de predicción en paralelo. La primera, llamada one-to-many, asigna múltiples anchors por objeto para generar una señal de entrenamiento que sea capaz de ayudar al modelo a aprender buenas representaciones. La segunda, llamada one-to-one, fuerza que cada objeto real tenga exactamente una predicción ganadora, entrenando al modelo para filtrar sus propios duplicados internamente.

    Durante la inferencia solo se usa la cabeza one-to-one. Como el modelo ya aprendió a producir una única predicción por objeto, no genera las múltiples cajas superpuestas que NMS tendría que limpiar. Esto resuelve directamente el problema de los clones, puesto que cada persona tiene su propia predicción sin que NMS tenga que intervenir y, en el proceso, descartar detecciones válidas.